In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

# --- Part 1: The Prunable Linear Layer ---
class PrunableLinear(nn.Module):
    def __init__(self, in_features, out_features):
        super(PrunableLinear, self).__init__()
        self.weight = nn.Parameter(torch.Tensor(out_features, in_features))
        self.bias = nn.Parameter(torch.Tensor(out_features))
        # Initialize gate scores to be high so gates start near 1.0
        self.gate_scores = nn.Parameter(torch.Tensor(out_features, in_features))

        nn.init.kaiming_uniform_(self.weight, a=np.sqrt(5))
        nn.init.constant_(self.bias, 0)
        nn.init.constant_(self.gate_scores, 2.0) # Start with gates ~0.88

    def forward(self, x):
        # Convert scores to 0-1 range
        gates = torch.sigmoid(self.gate_scores)
        # Element-wise multiplication (Hadamard product)
        pruned_weights = self.weight * gates
        return nn.functional.linear(x, pruned_weights, self.bias)

    def get_sparsity_loss(self):
        # L1 norm of the gates
        return torch.sum(torch.sigmoid(self.gate_scores))

# --- Model Definition ---
class SelfPruningNet(nn.Module):
    def __init__(self):
        super(SelfPruningNet, self).__init__()
        self.flatten = nn.Flatten()
        self.fc1 = PrunableLinear(32 * 32 * 3, 512)
        self.fc2 = PrunableLinear(512, 256)
        self.fc3 = PrunableLinear(256, 10)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.flatten(x)
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x

    def get_total_sparsity_loss(self):
        return self.fc1.get_sparsity_loss() + \
               self.fc2.get_sparsity_loss() + \
               self.fc3.get_sparsity_loss()

# --- Part 3: Training and Evaluation ---
def train_and_eval(lambd, epochs=5):
    # Data Loading
    transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
    trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
    trainloader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True)

    testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
    testloader = torch.utils.data.DataLoader(testset, batch_size=64, shuffle=False)

    model = SelfPruningNet()
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()
        for images, labels in trainloader:
            optimizer.zero_grad()
            outputs = model(images)
            class_loss = criterion(outputs, labels)
            sparsity_loss = model.get_total_sparsity_loss()

            total_loss = class_loss + lambd * sparsity_loss
            total_loss.backward()
            optimizer.step()

    # Evaluation
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in testloader:
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total

    # Calculate Sparsity
    total_weights = 0
    pruned_weights = 0
    all_gates = []

    for layer in [model.fc1, model.fc2, model.fc3]:
        gates = torch.sigmoid(layer.gate_scores).detach()
        all_gates.extend(gates.view(-1).tolist())
        total_weights += gates.numel()
        pruned_weights += (gates < 1e-2).sum().item()

    sparsity_pct = 100 * pruned_weights / total_weights
    return accuracy, sparsity_pct, all_gates

# --- Run Experiments ---
lambdas = [1e-5, 1e-4, 5e-4]
results = []

for l in lambdas:
    print(f"Running for Lambda: {l}")
    acc, sp, gates = train_and_eval(l)
    results.append((l, acc, sp, gates))

# --- Reporting ---
print("\n| Lambda | Test Accuracy | Sparsity Level (%) |")
print("|---|---|---|")
for res in results:
    print(f"| {res[0]} | {res[1]:.2f}% | {res[2]:.2f}% |")

# Plot Distribution for the best/medium model
plt.hist(results[1][3], bins=50, color='skyblue', edgecolor='black')
plt.title(f"Gate Distribution (Lambda={results[1][0]})")
plt.xlabel("Gate Value")
plt.ylabel("Frequency")
plt.show()

Running for Lambda: 1e-05


100%|██████████| 170M/170M [00:03<00:00, 43.6MB/s]
